In [0]:

CREATE OR REPLACE TABLE workspace.electiva.customers AS
SELECT * FROM read_files(
  '/Volumes/workspace/electiva/datos/customers.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.employee AS
SELECT * FROM read_files(
  '/Volumes/workspace/electiva/datos/employees.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.orders AS
SELECT * FROM read_files(
  '/Volumes/workspace/electiva/datos/orders.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.product_categories AS
SELECT * FROM read_files(
  '/Volumes/workspace/electiva/datos/productcategories.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.product_sub_categories AS
SELECT * FROM read_files(
  '/Volumes/workspace/electiva/datos/productsubcategories.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.products AS
SELECT * FROM read_files(
  '/Volumes/workspace/electiva/datos/products.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.vendor AS
SELECT * FROM read_files(
  '/Volumes/workspace/electiva/datos/vendors.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE workspace.electiva.vendor_product AS
SELECT * FROM read_files(
  '/Volumes/workspace/electiva/datos/vendorproduct.csv',
  format => 'csv', header => true, inferSchema => true
);

SHOW TABLES IN workspace.electiva;

-- ============================================================
-- PUNTO 1 (0.5)
-- user_name = customerid-XX-yy-AW
-- Ejemplo Catherine Abel (293) -> 293-CA-ab-AW
-- ============================================================

SELECT
  customerid,
  fullname,
  CONCAT(
    customerid, '-',
    UPPER(SUBSTRING(firstname, 1, 2)), '-',
    LOWER(SUBSTRING(lastname, 1, 2)),
    '-AW'
  ) AS user_name
FROM workspace.electiva.customers
ORDER BY customerid;

-- ============================================================
-- PUNTO 2 (1.0)
-- Tabla page_user con customerid y user_name
-- ============================================================

DROP TABLE IF EXISTS workspace.electiva.page_user;

CREATE TABLE workspace.electiva.page_user (
  customerid INT,
  user_name STRING
);

INSERT INTO workspace.electiva.page_user
SELECT
  customerid,
  CONCAT(
    customerid, '-',
    UPPER(SUBSTRING(firstname, 1, 2)), '-',
    LOWER(SUBSTRING(lastname, 1, 2)),
    '-AW'
  ) AS user_name
FROM workspace.electiva.customers;

SELECT *
FROM workspace.electiva.page_user
ORDER BY customerid;

-- ============================================================
-- PUNTO 3 (1.5)
-- Vista orders_agosto: pedidos de agosto 2011 + descuentos
-- Bikes 10%, Clothing 15%, Components 8%, Accessories 20%
-- ============================================================

CREATE OR REPLACE VIEW workspace.electiva.orders_agosto AS
WITH lineas AS (
  SELECT
    p.productname,
    p.modelname,
    u.user_name,
    o.salesorderid,
    c.categoryname,
    o.orderdate,
    o.linetotal,
    CASE
      WHEN c.categoryname ILIKE 'Bikes' THEN 0.10
      WHEN c.categoryname ILIKE 'Clothing' THEN 0.15
      WHEN c.categoryname ILIKE 'Components' THEN 0.08
      WHEN c.categoryname ILIKE 'Accessories' THEN 0.20
      ELSE 0
    END AS `descuento_mitad_año`,
    to_date(
      COALESCE(
        try_to_timestamp(CAST(o.orderdate AS STRING), 'M/d/yyyy'),
        try_to_timestamp(CAST(o.orderdate AS STRING))
      )
    ) AS fecha_orden
  FROM workspace.electiva.orders o
  INNER JOIN workspace.electiva.products p
    ON o.productid = p.productid
  INNER JOIN workspace.electiva.product_sub_categories sc
    ON p.subcategoryid = sc.subcategoryid
  INNER JOIN workspace.electiva.product_categories c
    ON sc.categoryid = c.categoryid
  INNER JOIN workspace.electiva.page_user u
    ON o.customerid = u.customerid
)
SELECT
  productname,
  modelname,
  user_name,
  salesorderid,
  categoryname,
  orderdate,
  linetotal,
  `descuento_mitad_año`,
  linetotal - linetotal * `descuento_mitad_año` AS total_descuentos_aplicados
FROM lineas
WHERE year(fecha_orden) = 2011
  AND month(fecha_orden) = 8;

SELECT *
FROM workspace.electiva.orders_agosto
ORDER BY salesorderid, productname;

-- ============================================================
-- PUNTO 4 (2.0)
-- Vista materializada customer_class
-- 3 ordenes = Oro | 2 = Plata | menos de 2 = Ocasional
-- ============================================================

CREATE OR REPLACE MATERIALIZED VIEW workspace.electiva.customer_class AS
WITH ordenes_2011 AS (
  SELECT
    customerid,
    COUNT(DISTINCT salesorderid) AS cantidad_ordenes
  FROM workspace.electiva.orders
  WHERE year(
    to_date(
      COALESCE(
        try_to_timestamp(CAST(orderdate AS STRING), 'M/d/yyyy'),
        try_to_timestamp(CAST(orderdate AS STRING))
      )
    )
  ) = 2011
  GROUP BY customerid
)
SELECT
  u.user_name,
  CASE
    WHEN o.cantidad_ordenes >= 3 THEN 'Oro'
    WHEN o.cantidad_ordenes = 2 THEN 'Plata'
    ELSE 'Ocasional'
  END AS user_level,
  c.customerid,
  c.fullname,
  o.cantidad_ordenes
FROM ordenes_2011 o
INNER JOIN workspace.electiva.customers c
  ON o.customerid = c.customerid
INNER JOIN workspace.electiva.page_user u
  ON c.customerid = u.customerid;

REFRESH MATERIALIZED VIEW workspace.electiva.customer_class;

SELECT *
FROM workspace.electiva.customer_class
ORDER BY cantidad_ordenes DESC, customerid;

SELECT user_level, COUNT(*) AS clientes
FROM workspace.electiva.customer_class
GROUP BY user_level
ORDER BY clientes DESC;
